Leaf Classification

The purpose of this task is to classify leaf images using supervised machine learning. This process includes preparing the data, analyzing the data and reviewing the analysis.

The dataset consists of three leaf classes: Basil, Lemon and Chinar. The original images contain RGB values which need to be converted to grayscale for the purpose of the analysis. The images are sourced from Kaggle.

To train the classifiers the following methods are used: Ridge Classifier, Random Forest and Multi-Layer Perceptron.

In [125]:
#All imports are done here

import numpy as np
import pandas as pd
import sklearn as sk
import matplotlib.pyplot as plt
import skimage as ski

#For future proof image handling instead of skimage.io
import imageio.v3 as iio

#For reading files from folders
from pathlib import Path

#Computer vision
import cv2 as cv


In [126]:
#Importing all images
basil = []
for file in Path('Basil').iterdir():
  if not file.is_file():
    continue

  basil.append(iio.imread(file))

chinar = []
for file in Path('Chinar').iterdir():
  if not file.is_file():
    continue

  chinar.append(iio.imread(file))

lemon = []
for file in Path('Lemon').iterdir():
  if not file.is_file():
    continue

  lemon.append(iio.imread(file))

In [127]:
#Converting all images to 3-bit grayscale
categories = [basil, chinar, lemon]
all_gray = []
all_red = []
all_green = []
all_blue = []

for leaves in categories:
  for leaf in leaves:
    grayLeaf = ski.color.rgb2gray(leaf)
    grayLeaf = np.uint8(np.clip(grayLeaf * 8, 0, 7))
    all_gray.append(grayLeaf)
    all_blue.append(leaf[:,:,0])
    all_green.append(leaf[:,:,1])
    all_red.append(leaf[:,:,2])

In [128]:
#Combining all images into one list and listing all the labels

#Images: 184, 120, 180
X = np.stack(all_gray)
X_width = X[:, ]

#Labels 0: basil, 1: chinar, 2: lemon
Y = np.concatenate([
    np.zeros(len(basil), dtype=int),
    np.ones(len(chinar), dtype=int),
    np.full(len(lemon), 2, dtype=int)
])

In [129]:
#Mean values for each image and each RGB channel
#Result is the average value for a single image and a single color

all_red = np.array(all_red)
red_mean = np.mean(all_red, axis=(1,2))

all_green = np.array(all_green)
green_mean = np.mean(all_green, axis=(1,2))


all_blue = np.array(all_blue)
blue_mean = np.mean(all_blue, axis=(1,2))


In [130]:
#Variance values for each image and each RGB channel
#Result is the average value for a single image and a single color

red_var = np.var(all_red, axis=(1,2))
green_var = np.var(all_green, axis=(1,2))
blue_var = np.var(all_blue, axis=(1,2))

In [131]:
#Gray-Level Co-Occurrence Matrix (GLCM)
#Correlation for GLCM means how similar each pixel is to it's neighbours.
#High correlation means there is less "texture" in the image so the whole image is more homogenous.

glcm_vertical_1 = []
glcm_vertical_2 = []
glcm_horizontal_1 = []
glcm_horizontal_2 = []

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[1], angles=[0], levels=8, symmetric=False, normed=True 
  )
  glcm_horizontal_1.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_horizontal_1 = np.array(glcm_horizontal_1)

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[2], angles=[0], levels=8, symmetric=False, normed=True 
  )
  glcm_horizontal_2.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_horizontal_2 = np.array(glcm_horizontal_2)

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[1], angles=[np.pi/2], levels=8, symmetric=False, normed=True 
  )
  glcm_vertical_1.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_vertical_1 = np.array(glcm_vertical_1)

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[2], angles=[np.pi/2], levels=8, symmetric=False, normed=True 
  )
  glcm_vertical_2.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_vertical_2 = np.array(glcm_vertical_2)

In [132]:
feature_vector = pd.concat([
  pd.DataFrame(Y),
  pd.DataFrame(red_mean),
  pd.DataFrame(red_var),
  pd.DataFrame(green_mean),
  pd.DataFrame(green_var),
  pd.DataFrame(blue_mean),
  pd.DataFrame(blue_var),
  pd.DataFrame(glcm_vertical_1),
  pd.DataFrame(glcm_vertical_2),
  pd.DataFrame(glcm_horizontal_1),
  pd.DataFrame(glcm_horizontal_2),
], axis=1)

feature_vector = feature_vector.set_axis([
  'leaf',
  'red_mean',
  'red_var',
  'green_mean', 
  'green_var', 
  'blue_mean', 
  'blue_var', 
  'glcm_vertical_1', 
  'glcm_vertical_2', 
  'glcm_horizontal_1', 
  'glcm_horizontal_2'
  ], axis=1)

In [133]:
feature_vector['glcm_mean'] = feature_vector[[
  'glcm_vertical_1', 
  'glcm_vertical_2', 
  'glcm_horizontal_1', 
  'glcm_horizontal_2'
  ]].mean(axis=1)

idx = (feature_vector.groupby('leaf')['glcm_mean'].transform(max) == feature_vector['glcm_mean'])
template_leaves = feature_vector[idx]

print(template_leaves['leaf'].index)

Index([12, 121, 139], dtype='int64')


In [ ]:
#Additional image-based features
basil_scores = []
chinar_scores = []
lemon_scores = []

basil_template = all_gray[template_leaves['leaf'].index[0]]
chinar_template = all_gray[template_leaves['leaf'].index[1]]
lemon_template = all_gray[template_leaves['leaf'].index[2]]

for leaf in all_gray:
  res = cv.matchTemplate(leaf, basil_template, cv.TM_CCOEFF_NORMED)
  basil_scores.append(res.max())

for leaf in all_gray:
  res = cv.matchTemplate(leaf, chinar_template, cv.TM_CCOEFF_NORMED)
  chinar_scores.append(res.max())

for leaf in all_gray:
  res = cv.matchTemplate(leaf, lemon_template, cv.TM_CCOEFF_NORMED)
  lemon_scores.append(res.max())

template_matching_scores = pd.DataFrame([basil_scores, chinar_scores, lemon_scores])

template_matching_scores = template_matching_scores.T
template_matching_scores.columns = ['basil_match', 'chinar_match', 'lemon_match']

feature_vector = pd.concat([feature_vector, template_matching_scores], axis=1)

In [139]:
print(feature_vector.columns)

feature_vector_complete = feature_vector.drop(['leaf', 'glcm_mean'], axis=1)

print(feature_vector_complete.columns)

Index(['leaf', 'red_mean', 'red_var', 'green_mean', 'green_var', 'blue_mean',
       'blue_var', 'glcm_vertical_1', 'glcm_vertical_2', 'glcm_horizontal_1',
       'glcm_horizontal_2', 'glcm_mean', 'basil_match', 'chinar_match',
       'lemon_match'],
      dtype='str')
Index(['red_mean', 'red_var', 'green_mean', 'green_var', 'blue_mean',
       'blue_var', 'glcm_vertical_1', 'glcm_vertical_2', 'glcm_horizontal_1',
       'glcm_horizontal_2', 'basil_match', 'chinar_match', 'lemon_match'],
      dtype='str')
